# Lab 03 - Data Preprocessing: Global Urban Air Quality and Pollution Time-Series

This notebook applies **Lab 3 - Data Preprocessing** to the selected assignment dataset. The focus is preparing a clean modeling table without editing the raw CSV.


## Lab 3 concepts used

- Parse and sort timestamp fields.
- Check missing values and duplicate rows.
- Impute missing numeric values.
- Encode categorical fields.
- Scale numeric fields.
- Engineer time features.
- Create a chronological train/test split.

Assignment guardrail: for the hazardous-event classifier, `European_AQI` is excluded from the primary feature set because it can leak the target definition.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
print('Shape:', data.shape)
print('\nMissing values:')
display(data.isna().sum().sort_values(ascending=False))
print('\nDuplicate rows:', data.duplicated().sum())
print('\nHazardous event distribution:')
display(data['Hazardous_Event'].value_counts(normalize=True).rename('proportion'))


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'PM2_5_ug_m3',
    'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
prepared = add_time_features(data)
prepared[['Timestamp', 'City', 'hour', 'dayofweek', 'month', 'dayofyear', 'is_weekend']].head()


In [ ]:
prepared['PM10_ug_m3'] = prepared['PM10_ug_m3'].fillna(
    prepared.groupby('City')['PM10_ug_m3'].transform('median')
)
prepared['PM10_ug_m3'] = prepared['PM10_ug_m3'].fillna(prepared['PM10_ug_m3'].median())
prepared.isna().sum().sort_values(ascending=False)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
train_df, test_df = chronological_split(prepared, train_size=0.8)
X_train = train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['Hazardous_Event']
X_test = test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['Hazardous_Event']

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print('Train period:', train_df['Timestamp'].min(), 'to', train_df['Timestamp'].max())
print('Test period:', test_df['Timestamp'].min(), 'to', test_df['Timestamp'].max())
print('Prepared train matrix:', X_train_prepared.shape)
print('Prepared test matrix:', X_test_prepared.shape)


In [ ]:
pd.DataFrame(X_train_prepared[:5], columns=preprocessor.get_feature_names_out()).head()


In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(data=prepared, x='hour', hue='Hazardous_Event')
plt.title('Hazardous events by hour after preprocessing')
plt.show()


## What was learned from Lab 3

The raw CSV remains unchanged. The modeling table now has parsed time fields, imputed `PM10_ug_m3`, encoded city values, scaled numeric fields, and a chronological split. This prepared structure is reused by later supervised-learning labs.
